# BattleNet Training on Kaggle GPU

**Dataset Structure:**
- Input (`/kaggle/input/`): Same as local `KIIT-MiTA/` - images in `train/images/`, YOLO labels in `train/labels/`
- Working (`/kaggle/working/`): Generated JSON annotations stored here

**Optimized hyperparameters:**
- Weight Decay: 1e-3 (+2.43% improvement)
- Batch Size: 32
- Augmentation: baseline (strong hurt performance)
- Loss: BCE (focal hurt -3.5%)

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Path Configuration

In [ ]:
# Kaggle paths
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Find dataset directory (the one with train/valid/test folders)
dataset_dirs = list(INPUT_ROOT.glob("*"))
DATASET_ROOT = None

print("Scanning input directories...")
for d in dataset_dirs:
    print(f"  - {d.name}")
    # Check if this is the KIIT-MiTA dataset
    if (d / "train" / "images").exists():
        DATASET_ROOT = d
        print(f"    -> Found dataset structure!")
        break

if DATASET_ROOT is None:
    raise ValueError("Could not find KIIT-MiTA dataset in /kaggle/input/")

print(f"\nDataset root: {DATASET_ROOT}")
print(f"\nDataset structure:")
for split in ['train', 'valid', 'test']:
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    num_images = len(list(images_dir.glob("*"))) if images_dir.exists() else 0
    num_labels = len(list(labels_dir.glob("*"))) if labels_dir.exists() else 0
    print(f"  {split}: {num_images} images, {num_labels} labels")

# Class names
CLASS_NAMES = ["Artilary", "Missile", "Radar", "M. Rocket Launcher", "Soldier", "Tank", "Vehicle"]
NUM_CLASSES = len(CLASS_NAMES)

## Convert YOLO Labels to JSON Format

YOLO format: `class_id center_x center_y width height` (normalized 0-1)

Convert to multi-label JSON format for training.

In [ ]:
def convert_yolo_to_json(split):
    """Convert YOLO TXT labels to JSON multi-label format."""
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    
    annotations = []
    
    # Get all image files
    image_files = sorted(list(images_dir.glob("*")))
    
    for img_path in tqdm(image_files, desc=f"Converting {split}"):
        # Get corresponding label file
        label_path = labels_dir / (img_path.stem + ".txt")
        
        # Create multi-hot label vector
        labels = np.zeros(NUM_CLASSES)
        class_ids = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                lines = f.readlines()
                
            for line in lines:
                line = line.strip()
                if line:
                    parts = line.split()
                    if len(parts) >= 1:
                        class_id = int(parts[0])
                        if 0 <= class_id < NUM_CLASSES:
                            labels[class_id] = 1
                            class_ids.append(class_id)
        
        # Create annotation entry
        rel_path = f"{split}/images/{img_path.name}"
        annotations.append({
            "image_path": rel_path,
            "label": labels.tolist(),
            "class_ids": sorted(class_ids)
        })
    
    return annotations

# Convert all splits and save to working directory
print("Converting YOLO labels to JSON format...")
for split in ['train', 'valid', 'test']:
    annotations = convert_yolo_to_json(split)
    output_path = WORK_DIR / f"{split}_annotations.json"
    
    with open(output_path, 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f"  {split}: {len(annotations)} annotations saved to {output_path.name}")

print("\nAnnotation files created in /kaggle/working/")

## Training Configuration

In [ ]:
CONFIG = {
    "architecture": "battlenet",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,

    # Batch / image
    "batch_size": 32,
    "image_size": 224,
    "num_workers": 4,

    # Training schedule — Phase 1 extended to 20 epochs (was still improving at 15)
    "epochs_head": 20,
    "epochs_finetune": 25,
    "lr_head": 1e-3,
    "lr_finetune": 1e-4,

    # Optimization — adam proven +2.43% vs adamw on this dataset
    "optimizer": "adam",
    "weight_decay": 1e-3,
    # Phase 1: cosine (eta_min raised to 1e-5, was killing LR too fast at 1e-6)
    # Phase 2: ReduceLROnPlateau on val_f1 (responds to plateaus, not fixed schedule)
    "scheduler_p1": "cosine",
    "scheduler_p2": "plateau",
    "clip_grad_norm": 1.0,

    # Augmentation / loss — proven optimal, do NOT change
    "augmentation": "baseline",
    "loss_type": "bce",
    "pos_weight": 1.0,

    # Architecture
    "fusion_dim": 768,
    "dropout": 0.3,
    "aux_loss_weight": 0.2,          # reduced from 0.3 — less auxiliary noise in Phase 2

    # Test-time augmentation
    "tta": True,                     # average original + h-flip at inference

    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "mixed_precision": torch.cuda.is_available(),
}

print("\n=== Training Configuration ===")
for k, v in CONFIG.items():
    print(f"{k}: {v}")

## Dataset and Dataloaders

In [ ]:
class KiitMitaDataset(torch.utils.data.Dataset):
    """KIIT-MiTA multi-label classification dataset."""
    
    def __init__(self, annotations_path, dataset_root, transform=None):
        with open(annotations_path, 'r') as f:
            self.annotations = json.load(f)
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.num_classes = NUM_CLASSES
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_path = self.dataset_root / ann["image_path"]
        image = Image.open(img_path).convert('RGB')
        
        # Labels already in multi-hot format
        labels = torch.tensor(ann["label"], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, labels

# Data transforms
def get_transforms(augmentation="baseline", image_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    return train_transform, val_transform

# Load datasets
train_ann_path = WORK_DIR / "train_annotations.json"
val_ann_path = WORK_DIR / "valid_annotations.json"
test_ann_path = WORK_DIR / "test_annotations.json"

train_transform, val_transform = get_transforms(CONFIG["augmentation"], CONFIG["image_size"])

train_dataset = KiitMitaDataset(train_ann_path, DATASET_ROOT, train_transform)
val_dataset = KiitMitaDataset(val_ann_path, DATASET_ROOT, val_transform)
test_dataset = KiitMitaDataset(test_ann_path, DATASET_ROOT, val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Verify label distribution
print("\nLabel distribution in training set:")
label_counts = np.zeros(NUM_CLASSES)
for i in range(len(train_dataset)):
    _, labels = train_dataset[i]
    label_counts += labels.numpy()

for i, count in enumerate(label_counts):
    print(f"  {CLASS_NAMES[i]:<20} {int(count)} samples")

## BattleNet Model Architecture

In [ ]:
import torch
import torch.nn as nn


class SqueezeExcitation(nn.Module):
    """Channel-wise SE recalibration."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)


class LabelCoOccurrence(nn.Module):
    """Learnable label co-occurrence module with learnable scale."""
    def __init__(self, num_classes):
        super().__init__()
        self.W = nn.Parameter(torch.eye(num_classes) + torch.randn(num_classes, num_classes) * 0.01)
        self.bias = nn.Parameter(torch.zeros(num_classes))
        # Learnable scale — initialized at 0.1 to match original behavior, clamped to prevent explosion
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, logits):
        scale = torch.clamp(self.scale, 0.0, 1.0)
        return logits + torch.matmul(logits, self.W.t()) * scale + self.bias


class BattleNetBackbone(nn.Module):
    """Dual-backbone: EfficientNet-V2-S + Swin-T."""
    def __init__(self, pretrained=True):
        super().__init__()

        # CNN Branch
        from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
        self.cnn = efficientnet_v2_s(
            weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.cnn_dim = self.cnn.classifier[-1].in_features
        self.cnn.classifier = nn.Identity()

        # Transformer Branch
        from torchvision.models import swin_t, Swin_T_Weights
        self.transformer = swin_t(
            weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.trans_dim = self.transformer.head.in_features
        self.transformer.head = nn.Identity()

    def forward(self, x):
        feat_cnn = self.cnn(x)
        feat_trans = self.transformer(x)
        return feat_cnn, feat_trans


class BattleNetClassifier(nn.Module):
    """BattleNet fusion head with auxiliary supervised branches.

    Improvements over v1:
    - Auxiliary heads on each backbone branch → direct gradient supervision, breaks
      the long chain bottleneck, regularizes each backbone independently
    - SE residual skip + BatchNorm → prevents SE from collapsing features to zero,
      normalizes recalibrated distribution
    - Learnable co-occurrence scale → model learns optimal correction strength
    """
    def __init__(self, cnn_dim, trans_dim, num_classes=7, fusion_dim=512, dropout=0.3):
        super().__init__()

        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
        )

        self.trans_proj = nn.Sequential(
            nn.Linear(trans_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
        )

        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_dim * 3, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.se = SqueezeExcitation(fusion_dim, reduction=16)
        self.se_bn = nn.BatchNorm1d(fusion_dim)   # BN after SE for stable distribution
        self.direct_head = nn.Linear(fusion_dim, num_classes)
        self.label_cooccur = LabelCoOccurrence(num_classes)

        # Auxiliary heads — one per backbone branch for direct supervision
        self.cnn_aux_head = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, num_classes),
        )
        self.trans_aux_head = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, num_classes),
        )

    def forward(self, backbone_output, return_aux=False):
        feat_cnn, feat_trans = backbone_output
        c = self.cnn_proj(feat_cnn)
        t = self.trans_proj(feat_trans)

        # Bilinear interaction: additive + multiplicative + difference signals
        fused_input = torch.cat([c + t, c * t, c - t], dim=-1)
        fused = self.fusion_mlp(fused_input)

        # SE recalibration with residual skip + BN
        fused = fused + self.se_bn(self.se(fused))

        logits = self.direct_head(fused)
        logits = self.label_cooccur(logits)

        if return_aux:
            return logits, self.cnn_aux_head(c), self.trans_aux_head(t)
        return logits


class BattleNet(nn.Module):
    """BattleNet dual-backbone hybrid fusion classifier."""
    def __init__(self, num_classes=7, pretrained=True, fusion_dim=512, dropout=0.3):
        super().__init__()
        self.backbone = BattleNetBackbone(pretrained=pretrained)
        self.classifier = BattleNetClassifier(
            cnn_dim=self.backbone.cnn_dim,
            trans_dim=self.backbone.trans_dim,
            num_classes=num_classes,
            fusion_dim=fusion_dim,
            dropout=dropout,
        )

    def forward(self, x, return_aux=False):
        backbone_out = self.backbone(x)
        return self.classifier(backbone_out, return_aux=return_aux)


# Create model
model = BattleNet(
    num_classes=CONFIG["num_classes"],
    pretrained=True,
    fusion_dim=CONFIG["fusion_dim"],
    dropout=CONFIG["dropout"],
).to(CONFIG["device"])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Fusion dim: {CONFIG['fusion_dim']} (was 512)")

## Training Setup

In [ ]:
# Loss function
criterion = nn.BCEWithLogitsLoss()

# Metrics computation
def compute_metrics(probs, labels):
    preds = (probs > 0.5).astype(np.float32)
    metrics = {}
    
    for i, class_name in enumerate(CONFIG["class_names"]):
        tp = np.sum((preds[:, i] == 1) & (labels[:, i] == 1))
        fp = np.sum((preds[:, i] == 1) & (labels[:, i] == 0))
        fn = np.sum((preds[:, i] == 0) & (labels[:, i] == 1))
        tn = np.sum((preds[:, i] == 0) & (labels[:, i] == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        metrics[class_name] = {"precision": precision, "recall": recall, "f1": f1}
    
    # Micro average
    total_tp = np.sum((preds == 1) & (labels == 1))
    total_fp = np.sum((preds == 1) & (labels == 0))
    total_fn = np.sum((preds == 0) & (labels == 1))
    
    metrics["micro"] = {
        "precision": total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0,
        "recall": total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0,
        "f1": 2 * total_tp / (2 * total_tp + total_fp + total_fn) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0,
    }
    
    metrics["exact_match"] = np.all(preds == labels, axis=1).mean()
    
    return metrics

# Training history
history = {"train_loss": [], "val_loss": [], "val_f1": []}

print("Training setup complete.")

## Training Loop

### Phase 1: Train Head Only (Backbone Frozen)

In [ ]:
device = torch.device(CONFIG["device"])
scaler = torch.cuda.amp.GradScaler() if CONFIG["mixed_precision"] else None

# Freeze backbone
for param in model.backbone.parameters():
    param.requires_grad = False

# Adam optimizer
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=CONFIG["lr_head"],
    weight_decay=CONFIG["weight_decay"],
)

# Cosine annealing — eta_min raised to 1e-5 (was 1e-6, was killing LR too fast)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["epochs_head"], eta_min=1e-5
)

best_val_f1 = 0.0

print(f"\n=== Phase 1: Training Head (Backbone Frozen) — {CONFIG['epochs_head']} epochs ===")
for epoch in range(CONFIG["epochs_head"]):
    # Train
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs_head']}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                logits, cnn_logits, trans_logits = model(images, return_aux=True)
                loss_main = criterion(logits, labels)
                loss_aux = 0.5 * criterion(cnn_logits, labels) + 0.5 * criterion(trans_logits, labels)
                w = CONFIG["aux_loss_weight"]
                loss = (1.0 - w) * loss_main + w * loss_aux
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), CONFIG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, cnn_logits, trans_logits = model(images, return_aux=True)
            loss_main = criterion(logits, labels)
            loss_aux = 0.5 * criterion(cnn_logits, labels) + 0.5 * criterion(trans_logits, labels)
            w = CONFIG["aux_loss_weight"]
            loss = (1.0 - w) * loss_main + w * loss_aux
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), CONFIG["clip_grad_norm"])
            optimizer.step()

        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    metrics = compute_metrics(all_probs, all_labels)

    # Record history
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_f1"].append(float(metrics["micro"]["f1"]))

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {metrics['micro']['f1']:.4f} | LR: {current_lr:.2e}")

    if metrics["micro"]["f1"] > best_val_f1:
        best_val_f1 = metrics["micro"]["f1"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "f1": best_val_f1,
            "config": CONFIG,
        }, WORK_DIR / "battlenet_head_best.pth")

print(f"\nPhase 1 complete. Best Val F1: {best_val_f1:.4f}")

### Phase 2: Fine-tune Entire Network

In [ ]:
# Unfreeze all parameters
for param in model.parameters():
    param.requires_grad = True

# Differential LRs: backbone 10x lower than fusion head
optimizer = optim.Adam(
    [
        {"params": list(model.backbone.parameters()), "lr": CONFIG["lr_finetune"] / 10},  # 1e-5
        {"params": list(model.classifier.parameters()), "lr": CONFIG["lr_finetune"]},      # 1e-4
    ],
    weight_decay=CONFIG["weight_decay"],
)

# ReduceLROnPlateau on val_f1 — decays LR when plateau detected, unlike fixed cosine
# patience=4: tolerate 4 epochs without improvement before halving LR
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-7, verbose=True
)

best_val_f1 = 0.0
start_epoch = CONFIG["epochs_head"]

print(f"\n=== Phase 2: Fine-tuning Entire Network — {CONFIG['epochs_finetune']} epochs ===")
print(f"Backbone LR: {CONFIG['lr_finetune'] / 10:.1e} | Classifier LR: {CONFIG['lr_finetune']:.1e}")

for epoch in range(CONFIG["epochs_finetune"]):
    # Train
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {start_epoch+epoch+1}/{start_epoch+CONFIG['epochs_finetune']}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                logits, cnn_logits, trans_logits = model(images, return_aux=True)
                loss_main = criterion(logits, labels)
                loss_aux = 0.5 * criterion(cnn_logits, labels) + 0.5 * criterion(trans_logits, labels)
                w = CONFIG["aux_loss_weight"]
                loss = (1.0 - w) * loss_main + w * loss_aux
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, cnn_logits, trans_logits = model(images, return_aux=True)
            loss_main = criterion(logits, labels)
            loss_aux = 0.5 * criterion(cnn_logits, labels) + 0.5 * criterion(trans_logits, labels)
            w = CONFIG["aux_loss_weight"]
            loss = (1.0 - w) * loss_main + w * loss_aux
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
            optimizer.step()

        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)

    # Validate
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    metrics = compute_metrics(all_probs, all_labels)
    val_f1 = metrics["micro"]["f1"]

    # Record history
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_f1"].append(float(val_f1))

    # Plateau scheduler steps on val_f1
    scheduler.step(val_f1)

    current_lr = optimizer.param_groups[1]["lr"]  # classifier LR
    print(f"  Epoch {start_epoch+epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | LR: {current_lr:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "epoch": start_epoch + epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "f1": best_val_f1,
            "config": CONFIG,
        }, WORK_DIR / "battlenet_finetune_best.pth")

print(f"\nPhase 2 complete. Best Val F1: {best_val_f1:.4f}")

## Test Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(WORK_DIR / "battlenet_finetune_best.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

# TTA transform: horizontal flip
tta_flip = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(p=1.0),  # always flip
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_with_tta(model, dataloader, use_tta=True):
    """Run inference, optionally averaging original + h-flip predictions."""
    model.eval()
    total_loss = 0.0
    all_probs, all_labels_list = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Testing"):
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)

            if use_tta:
                # Apply h-flip TTA: re-load batch with flip applied
                # images are already tensors — flip directly with torch.flip
                images_flip = torch.flip(images, dims=[-1])  # flip width axis
                logits_flip = model(images_flip)
                probs_flip = torch.sigmoid(logits_flip)
                probs = (probs + probs_flip) / 2.0

            all_probs.append(probs.cpu().numpy())
            all_labels_list.append(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss, np.concatenate(all_probs, axis=0), np.concatenate(all_labels_list, axis=0)

use_tta = CONFIG.get("tta", False)
test_loss, all_probs, all_labels = predict_with_tta(model, test_loader, use_tta=use_tta)
test_metrics = compute_metrics(all_probs, all_labels)

print(f"\n=== Test Results {'(with TTA)' if use_tta else ''} ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Exact Match Accuracy: {test_metrics['exact_match']:.4f}")
print(f"Micro F1:             {test_metrics['micro']['f1']:.4f}")
print(f"Micro Precision:      {test_metrics['micro']['precision']:.4f}")
print(f"Micro Recall:         {test_metrics['micro']['recall']:.4f}")

print("\nPer-Class F1 Scores:")
for class_name in CONFIG["class_names"]:
    print(f"  {class_name:<20} {test_metrics[class_name]['f1']:.4f}")

## Save Results

In [ ]:
# Save final results
results = {
    "config": CONFIG,
    "history": history,
    "test_metrics": {k: float(v) if isinstance(v, (np.float64, np.float32)) else v 
                     for k, v in test_metrics.items()},
    "test_loss": float(test_loss),
    "best_val_f1": float(best_val_f1),
}

with open(WORK_DIR / "battlenet_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {WORK_DIR / 'battlenet_results.json'}")
print(f"Best checkpoint: {WORK_DIR / 'battlenet_finetune_best.pth'}")
print(f"\nFiles in /kaggle/working/:")
for f in WORK_DIR.glob("*"):
    size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
    print(f"  {f.name:<30} {size_mb:>8.2f} MB")